In [1]:
# Run this cell to activate or reset the panel buttons

# Python libraries
import os
import sys
import pickle
import networkx as nx
from io import StringIO
%matplotlib inline
import matplotlib.pyplot as plt
plt.close()

# IPython
import ipywidgets as widgets
from qgridnext import show_grid
from IPython.display import display

# ROTIFER
import rotifer
from rotifer.db import ncbi
from rotifer.db import uniprot
from rotifer.genome import utils as rgu
from rotifer.core import functions as rcf
from rotifer.interval import utils as riu
from rotifer.genome.data import NeighborhoodDF
from rotifer.devel.beta import hmmer as rdbh
from rotifer.devel.beta import sequence as rdbs
from rotifer.devel.alpha import gian_func as rdag
from rotifer.devel.alpha import epsoares as rdae
from rotifer.devel.alpha import rodolfo as rdar

# Subroutines
def new_id():
    from datetime import datetime
    return datetime.now().strftime("%Y%m%d%H%M%S")

def new_session(widget):
    sesText.value = new_id()

def get_session_path():
    from pathlib import Path
    return Path(".rotifer", sesText.value)

def on_dropdown_change(change):
  if change['type'] == 'change' and change['name'] == 'value':
    sesText.value = change["new"]

def show_who(widget):
    with output:
        try:
            output.clear_output()
            get_ipython().run_line_magic('who','')
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

def show_logfile(widget):
    with output:
        try:
            output.clear_output()
            print("Session ID     :",sesText.value)
            get_ipython().run_line_magic('logstate','')
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

def show_connection(widget):
    with output:
        try:
            output.clear_output()
            print(get_jupyter_server_list_output())
            print("\nThis kernel's connection info:")
            get_ipython().run_line_magic('connect_info','')
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

def clean_output(widget):
    with output:
        try:
            output.clear_output()
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

def get_jupyter_server_list_output():
    server_loaders = []
    
    # Check for modern or legacy server utility packages
    try:
        from jupyter_server.serverapp import list_running_servers
        server_loaders.append(list_running_servers)
    except ImportError:
        pass
    try:
        from notebook.notebookapp import list_running_servers
        server_loaders.append(list_running_servers)
    except ImportError:
        pass

    active_servers = []
    for load_servers in server_loaders:
        for server in load_servers():
            base_url = server.get("url", "")
            token = server.get("token", "")
            root_dir = server.get("root_dir", server.get("notebook_dir", "Unknown Directory"))
            
            # Format URL with token authentication key if it exists
            if token:
                full_url = f"{base_url}?token={token}"
            else:
                full_url = base_url
                
            active_servers.append(f"{full_url} :: {root_dir}")

    # Standard terminal layout mirroring the CLI tool
    if active_servers:
        output_text = "Currently running servers:\n" + "\n".join(active_servers)
    else:
        output_text = (
            "Currently running servers:\n"
            "No active standalone web servers found.\n"
            "(If you are using VS Code Native layout, it executes via direct Python subprocesses.)"
        )
    return output_text

# Bind and show
def display_panel():
    buttons = widgets.HBox([saveButton, clsButton, whoButton, logButton, connButton, initButton])
    display(buttons)
    buttons2 = widgets.HBox([sesText,sesCombo,newButton])
    display(buttons2)
    display(output)

# Get data
def get_saved_sessions():
    from pathlib import Path
    sessions = []
    if os.path.exists(".rotifer"):
        if os.path.isdir(".rotifer"):
            sessions = [ item for item in Path('.rotifer').iterdir() if item.is_dir() and (item / "session.dat").exists() ]
        else:
            print(".rotifer exists but it is not a directory!!!")
    sessions = [ item.name for item in sorted(sessions, key=lambda x: (x / "session.dat").stat().st_mtime, reverse=True) ]
    return sessions

# Operate on data: load, save, etc
def initialize(widget):
    with output:
        output.clear_output()  # Clean previous click
        try:
            sesdir = get_session_path()
            if not sesdir.exists():
                print(f"Directory for session {sesText.value} created at {str(sesdir)}")
                sesdir.mkdir(parents=True, exist_ok=True)
            if get_ipython().logger.logfile == None:
                i = 0
                logfile = sesdir / f"log.{i}.py"
                while logfile.exists():
                    i = i + 1
                    logfile = sesdir / f"log.{i}.py"
                get_ipython().run_line_magic('logstart',str(logfile))
                del(logfile)
            rcf.restore_session(sesdir / "session")
            get_ipython().run_line_magic('who','')
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

def save_session(widget):
    with output:
        try:
            output.clear_output()
            path = get_session_path()
            path.mkdir(parents=True, exist_ok=True)
            rcf.save_session(path / "session")
            sesCombo.options = get_saved_sessions()
            sesCombo.value = sesText.value
            print(f"Session saved!")
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")

# Initialization before UI generation
_saved_sessions = get_saved_sessions()
if "_ROTIFER_SESSION_ID" not in globals():
    _ROTIFER_SESSION_ID = new_id()

# IPython widgets: create
output = widgets.Output()
newButton = widgets.Button(
    description="New session",
    button_style="success",
    icon="plus",
    tooltip="Create a new session"
)
initButton = widgets.Button(
    description="Reload",
    button_style="success",
    icon="refresh",
    tooltip="Replace data in memory with the last saved session"
)
saveButton = widgets.Button(
    description="Save",
    button_style="success",
    icon="floppy-o",
    tooltip="Save current session data"
)
whoButton  = widgets.Button(
    description="List",
    button_style="info",
    icon="list",
    tooltip="List variables, modules and functions"
)
logButton  = widgets.Button(
    description="Logstate",
    button_style="info",
    icon="history",
    tooltip="Show logging file name and info"
)
connButton = widgets.Button(
    description="Connection",
    button_style="info",
    icon="server",
    tooltip="Click to print the list of jupyter servers with auth tokens"
)
clsButton  = widgets.Button(
    description="Clear",
    button_style="info",
    icon="circle-thin",
    tooltip="Clear output"
)
sesCombo = widgets.Dropdown(
    options=_saved_sessions,
    description='Saved sessions:',
    disabled=False,
    style={'description_width': 'initial'},
    icon="check"
)
sesText = widgets.Text(
    placeholder='Rename current session',
    description='Session ID:',
    value=_saved_sessions[0] if len(_saved_sessions) > 0 else _ROTIFER_SESSION_ID,
    disabled=False
)

# Connect callbacks
newButton.on_click(new_session)
initButton.on_click(initialize)
saveButton.on_click(save_session)
whoButton.on_click(show_who)
logButton.on_click(show_logfile)
connButton.on_click(show_connection)
clsButton.on_click(clean_output)

# Attach the observer to the dropdown
sesCombo.observe(on_dropdown_change, names='value')

# Main
display_panel()

Output()

## Query selection

We have a list of candidates for exploring VirB4 taken from Comparative genomics of the FtsK–HerA superfamily of pumping ATPases: implications for the origins of chromosome segregation, cell division and viral capsid packaging

|Query Acc |	Source |	Alternative Acc |	Info |
|----------|-----------|--------------------|----------|
|P0A3W0.1 |	Uniprot |		|Golden annotated accession of VirB4 in Agrobacterium tumefaciens 	|
|NP_223210.1 |	NCBI |	15611559; Q9ZLT4 |	Cage_E Hp - Deemed classical VirB4 by Aravind FtsK paper|
|AAM08240.1 	|NCBI |	19919310;Q8RPE5 |	lvhB4 |
|NP_220495.1 |	NCBI| 	15603980;Q9ZE45 |	RP103 |	
|NP_541005.1 |	NCBI |	17988372;Q8YDZ4 |	BMEII0028 |	
|NP_863348.1 |	NCBI |	32469876;Q8GJA2 |	VirB4 Cje |	
|T18339 |	NCBI |	7465640;Q5ZYB6 |	Icmb_Lepn IcMB |	
|NP_388375.1 |	NCBI |	16077561;P96642 |	YddE_Bs YddE | 	
|AAP22573.1 |	NCBI |	37955734 |	CP81_Pae CP81 |	
|NP_058238.1 |	NCBI |	10957214 |	TrhC_St TrhC |	
|ZP_00026649.1 |	NCBI |	22980948 |	Reut5675_Reme|

# First search

We start by choosing one sequence and inspecting how many domains it possesses
In our case, VirB4 has two expected domains: an N-Terminal domain and an ATPase domain
We intend to do as many iterations as necessary on PSI-BLAST; however, to achieve better results, we need to split both domains and do separate searches for each

>important: it's not impossible to do a psiblast of the full protein and the library allows it, however, if accuracy is wanted, we don't recommend that procedure 

We chose as a starting point the protein Q9ZLT4, a cagE entry from _Helicobacter pylori_

We are going to organize our data into a dictionary structure, so it is simpler to access when the body of the analysis grow

Our first item in the dictionary will be a **sequence object** that will store the original sequence. The sequence object resembles a dataframe and can be operated as such.



## Loading the query

In [83]:
# Here we are creating the dictionary alongside the first entry of the dataframe
# Notice that the library has a loaded directory that we are selecting. In this case, it's the Uniprot database that is locally stored
# You can also pass a full path string to the local_database_path option

cagE = dict(data=rdbs.sequence(["Q9ZLT4"], local_database_path=os.path.join(rotifer.config["data"],"fadb","uniprot","uniprotDB"))) # the accession for the rdbs.sequence function must be on list format

local.easel: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.75s/it]


## Visualization

We can visualize the sequence using either `cagE['data'].view()` in a Jupyter notebook or `cagE['data']._view_sequence()` in a Jupyter console terminal window. 
Additionally, you can run also run `cagE['data'].view()` in a terminal window


## Splitting domains

To get a better understanding of what we are looking at and what the components of our query are, we should **split** the domains that are present in our protein when doing searches (psi-blast or jackhmmer ones)

By inspecting the Uniprot page or the Interpro database for TED markings, we can determine how many domain hits are detected on the target protein and where they occur. 
Additionally, we should also inspect the predicted protein structure, which can be either predicted (for example, AlphaFold DB) or experimentaly recovered (PDB entries)

Below, we can observe the structure that belongs to our Q9ZLT4 query:

### Todo: embed actual molsession here
![Molstar view](Screenshot_20261102_163234.png)

We know from the literature that Virb4 ATPase has an alpha-helical insertion
We also know the ATPase should contain:

- The motif GXXXXGK[TS], named Walker-A, after the first beta-sheet
- The motif hhhh[DE]E, named Walker-B (where "h" refers to a hydrophobic residue) on a betasheet
- One arginine residue located shortly after the Walker-B motif
- A sensor small residue (usually a glutamine) near the arginine residue

From the inspection we then decide on this:

- the N-terminal domain **starts at 1 and ends at 548**
- the Atpase domain **starts at 592 and ends at 891**


## Preparing our domain queries

In [ ]:
# psiblast search on the ATPase and N-terminal domains
cagE['qNterm'] = cagE["data"].slice((1,548)) # Here we use the slice function with the starting and ending positions and store it in a variable
cagE['qATP'] = cagE["data"].slice((592,891))

# Searching using psiblast

In [8]:
cagE["Ntermpsi1"] = rdar.psiblast(cagE["qNterm"], num_aln=5000, aln=False, db="/scratch/global/databases/fadb/uniprot/uniref50") # default parameters used, but number of alignments reported is asked to be 5000
cagE["Atpasepsi1"] = rdar.psiblast(cagE["qATP"], num_aln=5000, aln=False, db="/scratch/global/databases/fadb/uniprot/uniref50")
# aln = False option is used because we are searching a single sequence, we will set this to True when we create our first alignment and want to iterate

From this, we have a search for our N-Terminal domain and one for our ATPase domain stored in the dictionary keys
We have the hits table and the raw psiblast, accessible by [0] and [1], respectively

In [94]:
cagE["Atpasepsi1"][0].head(10) # Here we are looking at the first 10 results for the psiblast search of the ATPase domain

,hit,query,hitstart,hitend,evalue,querycoverage,querystart,queryend,iteration,bitscore,length
0,UniRef50_Q48252,Q9ZLT4,592,891,0.000000e+00,1.000,1,300,1,624.0,983
1,UniRef50_J0J980,Q9ZLT4,590,768,1.000000e-112,0.610,1,183,1,350.0,768
2,UniRef50_UPI0022B2529A,Q9ZLT4,22,175,3.000000e-110,0.513,147,300,1,328.0,267
3,UniRef50_UPI001F1056E8,Q9ZLT4,278,560,6.000000e-60,0.997,1,299,1,210.0,654
4,UniRef50_UPI000A4AF590,Q9ZLT4,70,352,1.000000e-57,0.997,1,299,1,199.0,445
5,UniRef50_A0A381F3V5,Q9ZLT4,547,829,2.000000e-55,0.997,1,299,1,201.0,923
6,UniRef50_Q9AGH2,Q9ZLT4,224,505,5.000000e-48,0.997,1,299,1,177.0,589
7,UniRef50_A0AA46RZP0,Q9ZLT4,551,831,3.000000e-46,0.993,3,300,1,175.0,922
8,UniRef50_Q9F245,Q9ZLT4,551,832,6.000000e-46,0.997,1,299,1,174.0,923
9,UniRef50_UPI001E4E2E51,Q9ZLT4,342,627,8.000000e-46,0.997,1,299,1,172.0,718


In [95]:
cagE["Ntermpsi1"][0].head(10) # Here we are looking at the first 10 results for the psiblast search of the N-Terminal domain

,hit,query,hitstart,hitend,evalue,querycoverage,querystart,queryend,iteration,bitscore,length
0,UniRef50_Q48252,Q9ZLT4,1,548,0.000000e+00,1.000,1,548,1,1114.0,983
1,UniRef50_J0J980,Q9ZLT4,1,546,0.000000e+00,0.996,3,548,1,1112.0,768
2,UniRef50_Q8GKD4,Q9ZLT4,1,209,1.000000e-145,0.381,287,495,1,425.0,209
3,UniRef50_E2CZ24,Q9ZLT4,1,106,2.000000e-66,0.193,93,198,1,218.0,106
4,UniRef50_UPI00356548C3,Q9ZLT4,1,67,1.000000e-35,0.122,1,67,1,136.0,68
5,UniRef50_J0JAR0,Q9ZLT4,1,67,3.000000e-35,0.122,1,67,1,135.0,90
6,UniRef50_A0ABQ0D5J5,Q9ZLT4,6,498,1.000000e-29,0.916,46,547,1,133.0,732
7,UniRef50_UPI002F90BC0E,Q9ZLT4,125,495,5.000000e-28,0.688,171,547,1,127.0,618
8,UniRef50_A0A381F3V5,Q9ZLT4,6,498,5.000000e-27,0.916,46,547,1,125.0,923
9,UniRef50_A0AAV6EEL4,Q9ZLT4,24,503,1.000000e-26,0.923,41,546,1,124.0,929


## Inspection of the first  psiblast search: ATPase

We  want to do psi-blast iterations. For this, we need to select sequences and create an alignment. This process will be repeated until we reach an iteration that slips out the intended targets. 

Often, it's useful to cluster our sequences, we can do this using mmseqs inside rotifer, however, it is important to remember that rdbs.sequence fetches the whole sequence, not only the domain we are looking at, so we are going to extract the envelope that was detected by psiblast with an added margin 

Additionally, as we are going to iterate, we can afford to be more rigid in our cutoffs for this initial analysis.  
It's worth noting that the ATPases we are working with have an alpha-helical insert and thus psi-blast algorithm might struggle and truncate otherwise perfectly valid hits

### How many results do we have?

In [103]:
cagE["Atpasepsi1"][0].hit.nunique() # We are using nunique so we know how many proteins do we have

1350

In [110]:
cagE["Atpasepsi1"][0].hit.count() # We can check how many hits are there and here we see that one protein harbored two hits from our query

1351

### How many hits survive our evalue cutoff of 0.1?

In [104]:
cagE["Atpasepsi1"][0].query('evalue <= 0.1').hit.nunique() # We lost 322 sequences

1028

In [114]:
#Let's check if that repeated hit is still here
cagE["Atpasepsi1"][0].query('evalue <= 0.1').hit.count() # The numbers match, one of the hits had an evalue larger than the cutoff we used

1028

## Loading our selected sequences

In [114]:
cagE['ATPaseseqs'] = rdbs.sequence(cagE["Atpasepsi1"][0].query('evalue <= 0.1').hit.to_list(), local_database_path='/scratch/global/databases/fadb/uniprot/uniref50')

local.easel: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1028/1028 [00:02<00:00, 356.03it/s]


In [ ]:
## It's interesting to check how our lengths are distributed 

In [115]:
cagE['ATPaseseqs'].to_hist() # A few number of sequences are making our histogram hard to read, let's remove them

Total proteins: 1028
count 	 sequence size
###############################################################################
██████████████████████████████████████████████                 444  49 - 704   
█████████████████████████████████████████████████████████████  577  704 - 1353 
                                                                 4  1353 - 2002
                                                                 0  2002 - 2651
                                                                 1  2651 - 3300
                                                                 0  3300 - 3949
                                                                 0  3949 - 4598
                                                                 0  4598 - 5247
                                                                 1  5247 - 5896
                                                                 1  5896 - 6545


/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:1140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = a.groupby('bin').agg({'count':'sum'}).reset_index().apply(tuple, axis=1)


In [117]:
cagE['ATPaseseqs'].filter(maxlen = 1500).to_hist()

Total proteins: 1021
count 	 sequence size
###############################################################################
███████████                                                     70  54 - 181   
██████████████                                                  88  181 - 308  
██████████████                                                  89  308 - 434  
█████████████                                                   78  434 - 560  
███████████████████                                            115  560 - 686  
██████████████████████████████                                 183  686 - 813  
█████████████████████████████████████████████████████████████  364  813 - 939  
███                                                             20  939 - 1065 
█                                                                8  1065 - 1192
█                                                                6  1192 - 1318


/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:1140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = a.groupby('bin').agg({'count':'sum'}).reset_index().apply(tuple, axis=1)


## Collecting the envelope
Even if we filter by evalue, we are subject to failures of the psi-blast algorithm. We **know** there is an insertion, and it can definitely fragment the hits. A useful check to decide how we expand is to check the max and minimum values from querystart and queryend on our psiblast results, this gives us an idea of how the envelope might be in need of expansion

In [123]:
cagE['Atpasepsi1'][0].querystart.max()

270

In [371]:
# Let's extract the envelope, we are being generous because the insertion
cagE['ATPaseenv'] = rdae.extract_envelope(cagE["Atpasepsi1"][0].query('evalue <= 0.1').rename(columns={'hit': 'sequence','length':'og_length'}), start ='hitstart', end = 'hitend', seqobj = cagE['ATPaseseqs'], expand = 270, local_database_path='/scratch/global/databases/fadb/uniprot/uniref50')

2026-11-06 01:12:40,203 ERROR rotifer.devel.beta.sequence Cannot compare sequence object and <class 'NoneType'>


In [ ]:
# We should check the alignment for the Walker-A and B motifs 
# Run this on a terminal
cagE['ATPaseenv']._view_sequence()

In [124]:
cagE['ATPaseenv'].to_hist() # also check the extraction length distribution

Total proteins: 1028
count 	 sequence size
###############################################################################
███                                                               30  41 - 78  
█████                                                             44  78 - 116 
██████                                                            48  116 - 154
█████                                                             40  154 - 191
████                                                              37  191 - 228
█████████████████████████████████████                            298  228 - 266
███████████████████████████████████████████████████████████████  498  266 - 304
███                                                               29  304 - 341
                                                                   1  341 - 378
                                                                   3  378 - 416


/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:1140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = a.groupby('bin').agg({'count':'sum'}).reset_index().apply(tuple, axis=1)


In [ ]:
# There's still some sequences that are , we should check if extending the alignment does something to them
cagE['ATPaseenv'] = rdag.extend_aln(cagE['ATPaseenv'],n_terminal=50,c_terminal=50,full_length_seqobj=cagE['ATPaseseqs'])

In [25]:
# Let's check if something drastic changed
cagE['ATPaseenv'].to_hist()

Total proteins: 1028
count 	 sequence size
###############################################################################
███                                                               26  55 - 101 
█████                                                             50  101 - 147
████                                                              43  147 - 194
█████                                                             45  194 - 240
█████                                                             46  240 - 286
████                                                              38  286 - 332
███████████████████████████████████████████████████████████████  545  332 - 378
██████████████████████████                                       227  378 - 425
                                                                   5  425 - 471
                                                                   3  471 - 517


/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:1140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = a.groupby('bin').agg({'count':'sum'}).reset_index().apply(tuple, axis=1)


In [ ]:
# We should check wheter the expected motifs are there or not
# Take these steps on either a jupyter console terminal or a OS terminal window

# First, visualize the alignment and understand which column(s) should not be empty

cagE['ATPaseenv']._view_sequence()

# You can also sort this based on column to aid visualization

cagE['ATPaseenv'].sort([241])._view_sequence() # The number in the parentheses is the number of the column




In [ ]:
# We can check how many sequences miss at least one of the conserved residues or both

In [39]:
# Walker A
cagE['ATPaseenv'].df.query('sequence.str[240] == "-"')['id'].count() # Notice the count is 0 based

177

In [38]:
# Walker B
cagE['ATPaseenv'].df.query('sequence.str[1537] == "-"')['id'].count()

66

In [40]:
# Both
cagE['ATPaseenv'].df.query('sequence.str[240] == "-" and sequence.str[1537] == "-"')['id'].count()

17

== Don't forget to check the stricture!!!==

In [49]:
cagE['ATPaseenv'] = cagE['ATPaseenv'].sort(['list'],id_list=['UniRef50_Q48252']).slice((226,1685)).align() # trimm down to the atpase domain, we need to try to get as close as possible

/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:601: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fields.append(identifiers.where(lambda x: x.isin(ids)).replace(ids).rename("list"))
FAMSA (Fast and Accurate Multiple Sequence Alignment) 
  version 2.2.3-5efa514 (2024-09-17)
  S. Deorowicz, A. Debudaj-Grabysz, A. Gudys

Aligning /dev/stdin
Params:
  no. of threads: 12
  guide tree method: single linkage (MST+Prim)
  guide tree speeding up heuristic: None
  distance measure: indel_div_lcs

Advanced params:
  no. of refinements: 100
  refinement threshold: 1000
  gap open cost (rescalled): -24425
  gap extension cost (rescalled): -2056
  gap terminal open cost (rescalled): -1085
  gap terminal extension cost (rescalled): -1085
  gap cost scaller

In [ ]:
# To aid our comprehension of our dataset, we can use mmseqs2 to clusterize our sequences

cagE['ATPaseenv'].add_cluster(cascade=[(100,100),(80,70),(80,50),(80,30),(0,0)], inplace=True)

In [71]:
# We then can check how many clusters are there
cagE['ATPaseenv'].df.nunique()

id                 1028
sequence           1014
length              208
type                  1
description        1028
full_length         514
start               299
end                 471
C_term              162
previous_length     206
c100i100           1014
c80i70              827
c80i50              472
c80i30              104
c0i0                  6
dtype: int64

In [72]:
cagE['ATPaseenv'].to_hist()

Total proteins: 1028
count 	 sequence size
###############################################################################
███                                                               30  41 - 79  
█████                                                             44  79 - 116 
██████                                                            48  116 - 154
█████                                                             40  154 - 191
████                                                              38  191 - 229
█████████████████████████████████████                            296  229 - 267
███████████████████████████████████████████████████████████████  503  267 - 304
███                                                               26  304 - 342
                                                                   0  342 - 379
                                                                   3  379 - 417


/home/luandreani/git/rotifer/lib/rotifer/devel/beta/sequence.py:1140: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test = a.groupby('bin').agg({'count':'sum'}).reset_index().apply(tuple, axis=1)


# Complementing our knowledge

Now that we have the clustering, it would also be usefull to understand what domains are present on our selection, taxonomic distribution and neighbourhood.  
We expect that VirB4 have a distribution where sequences from close species cluster toghether

## Identifying domains

In [ ]:
# We need to remove the gaps and pass hmmscan an ungapped list of sequences

In [76]:
#this is a temporary variable we can delete after 
ungap = cagE['ATPaseenv'].copy()
ungap.df['sequence'] = ungap.df.sequence.str.replace('-', '', regex=False)

# Note to self - add ungapping step on hmmscan functions

In [78]:
cagE['Atpasehs'] = rdbh.hmmscan(ungap, models_path="/scratch/global/databases/pfam/Pfam-A.hmm")

hmmscan: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1028/1028 [00:12<00:00, 84.13seq/s]


In [79]:
cagE['Atpasepfam'] = riu.architecture(cagE['Atpasehs'].query('c_evalue <= 0.1'), **riu.config['hmmer'])

## Identifying taxonomy

In [82]:
# First, let's extract the taxonomic IDs from the description

cagE["ATPaseenv"].df['taxid'] = cagE["ATPaseenv"].df.description.str.extract("TaxID=([0-9]+)")

In [ ]:
tc = ncbi.TaxonomyCursor() # Initializing the taxonomy cursor

cagE["ATPaseTax"] = tc.fetchall(cagE["ATPaseenv"].df.taxid.drop_duplicates().tolist()) # Getting the taxonomic information

### Taxonomy fetched
Here, it's possible to see information already separated for superkingdom, kingdom and phylum

We have lineage and classification columns that can bring different information (classification might have more items than lineage) and organism depends on the information provided for ncbi, sometimes it will have strain information

In [83]:
cagE["ATPaseTax"].head(5)

,taxid,organism,superkingdom,lineage,classification,alternative_taxids,kingdom,phylum
0,1005994,Trabulsiella guamensis ATCC 49490,Bacteria,bacteria>pseudomonadota>gammaproteobacteria>en...,Bacteria; Pseudomonadati; Pseudomonadota; Gamm...,1005994,Pseudomonadati,Pseudomonadota
1,1030,Thiothrix,Bacteria,bacteria>pseudomonadota>gammaproteobacteria>th...,Bacteria; Pseudomonadati; Pseudomonadota; Gamm...,1030,Pseudomonadati,Pseudomonadota
2,1034944,Legionella tunisiensis,Bacteria,bacteria>pseudomonadota>gammaproteobacteria>le...,Bacteria; Pseudomonadati; Pseudomonadota; Gamm...,1034944,Pseudomonadati,Pseudomonadota
3,104270,Tenacibaculum ovolyticum,Bacteria,bacteria>fcb group>bacteroidota>flavobacteriia...,Bacteria; Pseudomonadati; FCB group; Bacteroid...,104270,Pseudomonadati,Bacteroidota
4,1046,Chromatiaceae,Bacteria,bacteria>pseudomonadota>gammaproteobacteria>ch...,Bacteria; Pseudomonadati; Pseudomonadota; Gamm...,1046,Pseudomonadati,Pseudomonadota


In [84]:
# Let's add the taxonomy to our sequence object
cagE["ATPaseenv"].df = cagE["ATPaseenv"].df.merge(cagE["ATPaseTax"], on="taxid", how="left")

# Deciding which sequences will be included on the alignment

We now have information to decide which sequences might go to the alignment that will be the base for a second psi-blast iteration

We need:

- Reduced redundancy
- Sequences that possess the expected motifs
- Preserve taxonomic diversity

The only thing missing would be a phylogeny which we can compare our clustering to.

You unfortunately still can't call tree software from inside ROTIFER, but exporting the alignment is relatively easy

In [85]:
# Exporting
cagE["ATPaseenv"].to_file('Atpasepsi1aln.fasta')

1028

We can then run FastTree on the 1028 sequences (we are using FastTree to get a fast exploratory tree)

`FastTree Atpasepsi1aln.fasta > Atpasepsi1.fasttree.nwk`

Notice we haven't removed the sequences which are missing a conserved motif, this is a decision made here as we find it could be useful to know how far off the incomplete sequences fall

## Other information to be gathered

While we wait for the phylogeny to resolve, there are preliminar analysis that can still be done so we understand our dataset better

In [93]:
cagE["ATPaseenv"].df.superkingdom.value_counts()

superkingdom
Bacteria                972
                         20
Archaea                  14
unclassified entries     12
Eukaryota                 9
Viruses                   1
Name: count, dtype: int64

In [95]:
cagE["ATPaseenv"].df.kingdom.value_counts()

kingdom
Pseudomonadati       795
Bacillati             75
Fusobacteriati        23
Thermoproteati         8
Nanobdellati           5
Thermotogati           4
Methanobacteriati      1
Name: count, dtype: int64

In [94]:
cagE["ATPaseenv"].df.phylum.value_counts()

phylum
Pseudomonadota              563
Campylobacterota             49
Bacillota                    44
Bacteroidota                 42
Actinomycetota               23
Fusobacteriota               23
Thermodesulfobacteriota      12
Acidobacteriota              10
Eukaryota                     9
Spirochaetota                 9
Gemmatimonadota               5
Aquificota                    5
Bdellovibrionota              4
Synergistota                  4
Thermoproteota                4
Myxococcota                   3
Iainarchaeota                 3
Planctomycetota               2
Nitrososphaerota              2
Bathyarchaeota                2
Microcaldota                  2
Nitrospirota                  2
Chloroflexota                 2
Acidulodesulfobacteriota      2
Binatota                      2
Chlamydiota                   2
Chlorobiota                   2
Uroviricota                   1
Methanobacteriota             1
Roizmaniibacteriota           1
Deferribacterota              1
A

In [98]:
cagE["ATPaseenv"].df.groupby('c80i50')['phylum'].value_counts()

c80i50                  phylum           
UniRef50_A0A011TLE1     Pseudomonadota       2
UniRef50_A0A060C9F1     Pseudomonadota       2
UniRef50_A0A062VE17     Pseudomonadota       1
UniRef50_A0A067W9N2     Pseudomonadota       1
UniRef50_A0A075H9T4     Methanobacteriota    1
                                            ..
UniRef50_UPI00406A5B8F  Pseudomonadota       4
UniRef50_UPI00406D1DCA  Pseudomonadota       1
UniRef50_V8CE48         Campylobacterota     3
UniRef50_W0GH52         Campylobacterota     1
UniRef50_W0N2Q7         Pseudomonadota       1
Name: count, Length: 465, dtype: int64

In [ ]:
# Creating our first alignment
cagE['ATPasec0i01'] = cagE['ATPaseenv'].filter(f"id in {cagE['ATPaseenv'].df.query('c0i0 == "UniRef50_A0AAW9DAR3/433-671"').sort_values('evalue').groupby('c80i50').head(2).id.tolist()}").align().sort(['evalue']).filter('sequence.str[9] != "-"').align().sort(['evalue']).filter('~c80i50.duplicated()').align()
cagE['ATPasec0i01'] = cagE['ATPasec0i01'].filter('sequence.str[454] != "-"').filter(remove = ['UniRef50_UPI0029C012DE/63-279','UniRef50_A0A1R3L3S7/46-244']).align().slice((2,-2)).sort(['evalue'])

# N-Terminal Region

In [382]:
cagE['Ntermseqs'] = rdbs.sequence(cagE["Ntermpsi1"][0].query('evalue <= 0.1').hit.to_list(), local_database_path='/scratch/global/databases/fadb/uniprot/uniref50')

local.easel: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 229/229 [00:03<00:00, 67.88it/s]


In [362]:
cagE["Ntermpsi1"][0].query('evalue <= 0.1').hit.nunique()

229

In [363]:
cagE["Ntermpsi1"][0].query('evalue <= 0.1').hit.count()

229

In [365]:
cagE["Ntermpsi1"][0].query('evalue <= 0.1').queryend.max()

548

In [368]:
cagE["Ntermpsi1"][0].query('evalue <= 0.1').querystart.value_counts()

querystart
167    104
171      9
175      9
172      7
319      7
187      5
173      4
1        3
163      3
154      3
174      3
169      3
149      3
35       3
397      3
176      2
315      2
170      2
188      2
290      2
385      2
375      2
157      2
130      2
166      2
46       2
177      2
158      2
143      1
178      1
41       1
184      1
327      1
161      1
44       1
39       1
457      1
212      1
153      1
373      1
287      1
452      1
226      1
93       1
370      1
155      1
53       1
168      1
293      1
334      1
40       1
57       1
3        1
183      1
152      1
61       1
313      1
141      1
148      1
286      1
186      1
392      1
Name: count, dtype: int64

In [383]:
cagE['NTermenv'] = rdae.extract_envelope(cagE["Ntermpsi1"][0].query('evalue <= 0.1').rename(columns={'hit': 'sequence','length':'og_length'}), start ='hitstart', end = 'hitend', seqobj = cagE['Ntermseqs'], expand = 10, local_database_path='/scratch/global/databases/fadb/uniprot/uniref50')

2026-11-06 01:27:48,131 ERROR rotifer.devel.beta.sequence Cannot compare sequence object and <class 'NoneType'>
